# Apex Retail Intelligence

## Notebook 1 : Raw & Landing Layer

### Objective

The main objective of this notebook is to perform the first two phases of the Medallion Architecture.

In this notebook, I will:

- Read all historical datasets.
- Read all incremental datasets.
- Store the raw data.
- Convert the raw CSV files into Parquet format.
- Validate the data using audit files.
- Generate a PASS / FAIL report before moving to the Bronze layer.

### Technologies Used
- PySpark
- Databricks
- CSV
- Parquet

### Expected Outcome

After completing this notebook, all datasets will be available in the Landing Layer and will be ready for further processing in the Bronze Layer.

## Project Pipeline

```text
                Apex Retail Intelligence Pipeline

 Historical CSV          Incremental CSV
        │                      │
        └──────────┬───────────┘
                   ▼
            Raw Landing Layer
                   │
                   ▼
             Bronze Layer
            (Delta Storage)
                   │
                   ▼
              Silver Layer
      (Cleaning + MERGE + SCD)
                   │
                   ▼
               Gold Layer
      (Dimensions & Fact Tables)
                   │
                   ▼
              KPI Dashboard
```

In [0]:
# ================================================================
# Apex Retail Intelligence
# Notebook 1 : Raw & Landing Layer
# ================================================================

# Import commonly used Spark SQL functions.
# These functions will be used throughout the project.

from pyspark.sql.functions import *

# Import Spark SQL data types.
# These data types will be useful while transforming columns
# in the Silver layer.

from pyspark.sql.types import *

# Import datetime module.
# It will help us record timestamps whenever required.

from datetime import datetime

print("Libraries imported successfully.")

Libraries imported successfully.


## Verify Spark Environment

Before reading any dataset, it is a good practice to verify that the Spark session is working properly.

In this step, I will check:

- Spark Version
- Current User
- Current Working Directory

This helps in confirming that the Databricks environment is ready.

In [0]:
# Display the current Spark version.

print("Spark Version :", spark.version)

Spark Version : 4.1.0


In [0]:
# Display the username of the current Databricks workspace.

print("Current User :", spark.sql("SELECT current_user()").first()[0])

Current User : akashpatra788@gmail.com


In [0]:
# Import os module.
# This module helps us interact with the operating system.

import os

# Display the current working directory.

print("Current Working Directory :")
print(os.getcwd())

Current Working Directory :
/Workspace/Users/akashpatra788@gmail.com/Apex_Retail_Intelligence/01_Raw_Landing


## Check Available File System

Before loading the datasets, I will check the available file system.

This step helps me understand where the datasets are stored inside Databricks.

In [0]:
# Display the root directories available in Databricks.

display(dbutils.fs.ls("/"))

path,name,size,modificationTime
dbfs:/Volumes/,Volumes/,0,0
dbfs:/Workspace/,Workspace/,0,0
dbfs:/databricks-datasets/,databricks-datasets/,0,0


## Step 1 : Define Dataset Locations

Before reading the datasets, I will define the location of all CSV files.

Keeping all file paths in one place makes the notebook easier to manage.

If the dataset location changes in future, I only need to update this section.

In [0]:
# List the Workspace folder

display(dbutils.fs.ls("dbfs:/Workspace/Users/akashpatra788@gmail.com/Apex_Retail_Intelligence/datasets"))

path,name,size,modificationTime
dbfs:/Workspace/Users/akashpatra788@gmail.com/Apex_Retail_Intelligence/datasets/audit/,audit/,4096,1786183264550
dbfs:/Workspace/Users/akashpatra788@gmail.com/Apex_Retail_Intelligence/datasets/historical/,historical/,4096,1786183264550
dbfs:/Workspace/Users/akashpatra788@gmail.com/Apex_Retail_Intelligence/datasets/incremental/,incremental/,4096,1786183264550


### Why am I doing this?

Before loading any dataset, it is a good practice to keep all file paths in one place.

This makes the notebook easy to maintain. If the dataset location changes in the future, I only need to update the path here instead of changing it throughout the notebook.

In [0]:
# ============================================================
# Step 1 : Define Dataset Paths
# ============================================================

# Store the common project paths.
# All datasets will be read from the Unity Catalog Volume.

RAW_PATH = "/Volumes/workspace/default/apex_retail_volume/raw"

LANDING_PATH = "/Volumes/workspace/default/apex_retail_volume/landing"

BRONZE_PATH = "/Volumes/workspace/default/apex_retail_volume/bronze"

SILVER_PATH = "/Volumes/workspace/default/apex_retail_volume/silver"

GOLD_PATH = "/Volumes/workspace/default/apex_retail_volume/gold"


# ---------------- Historical Datasets ----------------

customer_historical_path = f"{RAW_PATH}/historical/customer_historical.csv"

product_historical_path = f"{RAW_PATH}/historical/product_historical.csv"

sales_historical_path = f"{RAW_PATH}/historical/sales_historical.csv"


# ---------------- Incremental Datasets ----------------

customer_incremental_path = f"{RAW_PATH}/incremental/customer_incremental.csv"

product_incremental_path = f"{RAW_PATH}/incremental/product_incremental.csv"

sales_incremental_path = f"{RAW_PATH}/incremental/sales_incremental.csv"


# ---------------- Audit Datasets ----------------

customer_historical_audit_path = (
    f"{RAW_PATH}/audit/customer_historical_audit.csv"
)

customer_incremental_audit_path = (
    f"{RAW_PATH}/audit/customer_incrementalaudit.csv"
)

product_historical_audit_path = (
    f"{RAW_PATH}/audit/product_historical_audit.csv"
)

product_incremental_audit_path = (
    f"{RAW_PATH}/audit/product_incrementalaudit.csv"
)

sales_historical_audit_path = (
    f"{RAW_PATH}/audit/sales_historical_audit.csv"
)

sales_incremental_audit_path = (
    f"{RAW_PATH}/audit/sales_incrementalaudit.csv"
)

print("All dataset paths have been defined successfully.")

All dataset paths have been defined successfully.


## Step 2 : Check Dataset Folders

In this step, I am checking whether all the required dataset folders are available.

The project contains three types of datasets:

- Historical Data
- Incremental Data
- Audit Data

If all folders are available, I can proceed to read the CSV files in the next step.

In [0]:
# ============================================================
# Step 2 : Check Available Files
# ============================================================

# Display all files available in the Raw Layer.

print("Historical Folder")

display(dbutils.fs.ls(f"{RAW_PATH}/historical"))

print("Incremental Folder")

display(dbutils.fs.ls(f"{RAW_PATH}/incremental"))

print("Audit Folder")

display(dbutils.fs.ls(f"{RAW_PATH}/audit"))

Historical Folder


path,name,size,modificationTime
dbfs:/Volumes/workspace/default/apex_retail_volume/raw/historical/customer_historical.csv,customer_historical.csv,82586,1786189557000
dbfs:/Volumes/workspace/default/apex_retail_volume/raw/historical/product_historical.csv,product_historical.csv,128050,1786189556000
dbfs:/Volumes/workspace/default/apex_retail_volume/raw/historical/sales_historical.csv,sales_historical.csv,120507,1786189556000


Incremental Folder


path,name,size,modificationTime
dbfs:/Volumes/workspace/default/apex_retail_volume/raw/incremental/customer_incremental.csv,customer_incremental.csv,106981,1786189622000
dbfs:/Volumes/workspace/default/apex_retail_volume/raw/incremental/product_incremental.csv,product_incremental.csv,139293,1786189622000
dbfs:/Volumes/workspace/default/apex_retail_volume/raw/incremental/sales_incremental.csv,sales_incremental.csv,115446,1786189622000


Audit Folder


path,name,size,modificationTime
dbfs:/Volumes/workspace/default/apex_retail_volume/raw/audit/customer_historical_audit.csv,customer_historical_audit.csv,48,1786189665000
dbfs:/Volumes/workspace/default/apex_retail_volume/raw/audit/customer_incrementalaudit.csv,customer_incrementalaudit.csv,49,1786189666000
dbfs:/Volumes/workspace/default/apex_retail_volume/raw/audit/customer_incrementalaudit_silver.csv,customer_incrementalaudit_silver.csv,41,1786189665000
dbfs:/Volumes/workspace/default/apex_retail_volume/raw/audit/customer_silver_audit.csv,customer_silver_audit.csv,48,1786189666000
dbfs:/Volumes/workspace/default/apex_retail_volume/raw/audit/product_historical_audit.csv,product_historical_audit.csv,47,1786189666000
dbfs:/Volumes/workspace/default/apex_retail_volume/raw/audit/product_incrementalaudit.csv,product_incrementalaudit.csv,48,1786189666000
dbfs:/Volumes/workspace/default/apex_retail_volume/raw/audit/product_incrementalaudit_silver.csv,product_incrementalaudit_silver.csv,40,1786189665000
dbfs:/Volumes/workspace/default/apex_retail_volume/raw/audit/product_silver_audit.csv,product_silver_audit.csv,47,1786189665000
dbfs:/Volumes/workspace/default/apex_retail_volume/raw/audit/sales_historical_audit.csv,sales_historical_audit.csv,45,1786189665000
dbfs:/Volumes/workspace/default/apex_retail_volume/raw/audit/sales_incrementalaudit.csv,sales_incrementalaudit.csv,46,1786189665000


## Step 3 : Read Historical Datasets

In this step, I am loading the historical customer, product and sales datasets into PySpark DataFrames.

These datasets contain the initial data that will be used to build the Bronze, Silver and Gold layers.

In [0]:
# ============================================================
# Step 3 : Read Historical CSV Files
# ============================================================

# Read the Customer Historical Dataset.
# Header is enabled because the first row contains column names.
# All columns are loaded as String type.

customer_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", False)
    .csv(customer_historical_path)
)

# Read the Product Historical Dataset.

product_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", False)
    .csv(product_historical_path)
)

# Read the Sales Historical Dataset.

sales_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", False)
    .csv(sales_historical_path)
)

print("Historical datasets loaded successfully.")

Historical datasets loaded successfully.


## Step 3 : Verify Dataset Files

In this step, I am checking whether all the required CSV files are available in their respective folders.

This helps avoid file path errors before reading the datasets.

In [0]:
# ============================================================
# Step 3 : Verify Dataset Files
# ============================================================

# Display all files available in the Raw folders.

print("Historical Files")
historical_files = [file.name for file in dbutils.fs.ls(f"{RAW_PATH}/historical")]
print(historical_files)

print("\nIncremental Files")
incremental_files = [file.name for file in dbutils.fs.ls(f"{RAW_PATH}/incremental")]
print(incremental_files)

print("\nAudit Files")
audit_files = [file.name for file in dbutils.fs.ls(f"{RAW_PATH}/audit")]
print(audit_files)

Historical Files
['customer_historical.csv', 'product_historical.csv', 'sales_historical.csv']

Incremental Files
['customer_incremental.csv', 'product_incremental.csv', 'sales_incremental.csv']

Audit Files
['customer_historical_audit.csv', 'customer_incrementalaudit.csv', 'customer_incrementalaudit_silver.csv', 'customer_silver_audit.csv', 'product_historical_audit.csv', 'product_incrementalaudit.csv', 'product_incrementalaudit_silver.csv', 'product_silver_audit.csv', 'sales_historical_audit.csv', 'sales_incrementalaudit.csv', 'sales_incrementalaudit_silver.csv', 'sales_silver_audit.csv']


## Step 4 : Read Historical Datasets

Now I am loading the historical datasets into PySpark DataFrames.

At this stage, I am only reading the data.

No cleaning or transformation is performed in this step.

In [0]:
# ============================================================
# Step 4 : Read Historical Datasets
# ============================================================

# Read Customer Historical Dataset

customer_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", False)
    .csv(customer_historical_path)
)

# Read Product Historical Dataset

product_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", False)
    .csv(product_historical_path)
)

# Read Sales Historical Dataset

sales_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", False)
    .csv(sales_historical_path)
)

print("Historical datasets loaded successfully.")

Historical datasets loaded successfully.


## Step 5 : Preview Historical Datasets

After loading the datasets, I am displaying a few records from each dataset.

This helps me verify that the files have been loaded successfully.

In [0]:
# ============================================================
# Step 5 : Preview Historical Datasets
# ============================================================

# Display Customer Historical Dataset

display(customer_df)

# Display Product Historical Dataset

display(product_df)

# Display Sales Historical Dataset

display(sales_df)

customer_id,age,gender,income_bracket,loyalty_program,membership_years,churned,marital_status,number_of_children,education_level,occupation,customer_zip_code,customer_city,customer_state
1,56,Other,High,No,0,No,Divorced,3,Bachelor's,Self-Employed,37848,City D,State Y
2,69,Female,Medium,No,2,No,Married,2,PhD,Unemployed,44896,null,State X
3,46,Female,Low,No,5,No,Married,3,Bachelor's,Self-Employed,11816,City B,State X
4,32,Female,Low,No,0,No,Divorced,2,Master's,Employed,78604,City A,State Y
5,60,Female,null,Yes,7,Yes,Divorced,2,Bachelor's,Employed,17760,City B,State Z
6,25,Other,Medium,Yes,4,Yes,Divorced,0,Bachelor's,Unemployed,54549,City D,State Z
7,78,Male,High,No,0,No,Single,2,Master's,Retired,76235,City D,State X
8,38,null,Low,Yes,2,No,Married,1,Master's,Employed,52863,City B,State Y
9,56,Female,Low,No,0,No,Single,4,PhD,Self-Employed,65537,City C,State Y
10,75,Male,Medium,No,3,No,Married,2,High School,Self-Employed,43331,City C,State X


product_id,product_name,product_brand,product_category,product_rating,product_review_count,product_stock,product_return_rate,product_size,product_weight,product_color,product_material,product_manufacture_date,product_expiry_date,product_shelf_life,unit_price
1480,Product D,Brand Y,Electronics,2.5,560,48,0.4,Small,4.61,Red,Metal,2019-08-04 01:47:01,2022-05-28 14:54:02,250,49.72
1597,Product C,Brand X,Groceries,4.7,413,80,0.3,Medium,0.84,Blue,Metal,2019-10-23 19:59:17,2022-12-19 08:04:41,180,817.76
5142,Product B,null,Toys,4.6,312,14,0.08,Medium,0.23,Green,Plastic,2018-05-12 08:00:29,2023-02-01 12:15:07,131,270.3
8447,Product A,Brand Z,Toys,1.1,110,69,0.09,Large,4.37,Blue,Wood,2019-11-15 16:17:29,2023-02-05 11:46:57,16,547.84
6025,Product C,Brand X,Clothing,3.8,172,25,0.39,Small,1.68,Red,Metal,2019-08-27 02:58:19,2023-10-05 08:13:07,57,785.29
1883,Product A,Brand Y,Clothing,3.3,678,74,0.38,Large,0.26,null,Glass,2019-02-09 22:23:36,2022-10-31 01:48:25,89,751.17
7781,Product D,Brand Y,Toys,2.4,434,20,0.09,Medium,9.12,White,Plastic,2018-12-14 12:35:24,2023-11-04 17:03:25,316,340.07
8642,Product B,Brand Y,Groceries,2.4,868,60,0.26,Medium,0.58,Blue,Wood,2018-01-06 22:13:14,2022-01-24 05:50:56,360,17.7
7193,Product B,Brand Z,Groceries,null,392,45,0.28,Medium,6.52,Blue,Wood,2018-12-25 09:49:50,2022-10-04 03:00:26,77,458.85
7739,Product A,Brand X,Clothing,3.6,356,61,0.03,Large,8.38,Red,Plastic,2018-02-27 19:00:28,2022-02-20 12:19:26,10,612.28


transaction_id,transaction_date,customer_id,product_id,quantity,unit_price,discount_applied,payment_method,store_location,transaction_hour,day_of_week,week_of_year,month_of_year,total_sales,promotion_id,promotion_type,holiday_season,season,weekend
503290,2020-10-11 10:08:52,1,1480,8,49.72,0.5,Credit Card,Location A,18,Wednesday,27,7,563.16,271,20% Off,No,Spring,Yes
347796,2021-12-08 01:07:40,2,1597,7,817.76,0.32,Credit Card,Location C,15,Friday,20,2,7554.57,631,Flash Sale,No,Summer,Yes
493688,2020-02-17 09:40:48,3,5142,8,270.3,0.35,Debit Card,Location A,9,Saturday,35,6,7564.14,879,Flash Sale,Yes,Winter,Yes
861348,2020-08-13 00:43:14,4,8447,2,547.84,0.1,null,Location A,13,Friday,42,8,8125.92,211,Buy One Get One Free,Yes,Winter,No
535835,2021-07-02 11:59:03,5,6025,4,785.29,0.17,Mobile Payment,Location C,17,Monday,37,3,114.32,862,Flash Sale,Yes,Summer,Yes
978720,2020-12-01 16:31:54,6,1883,2,751.17,0.32,Credit Card,Location B,19,Saturday,50,2,3372.17,171,Buy One Get One Free,Yes,Summer,No
24070,2020-03-14 22:58:48,7,7781,5,340.07,0.41,Debit Card,Location A,15,Thursday,12,10,1322.64,664,Buy One Get One Free,No,Spring,No
752282,2021-11-17 15:56:04,8,8642,7,17.7,0.42,Debit Card,Location D,22,Saturday,3,9,1716.65,733,Buy One Get One Free,Yes,Summer,Yes
11898,2020-05-18 08:48:18,9,7193,7,458.85,0.2,Mobile Payment,Location B,19,Sunday,6,10,1358.62,429,Flash Sale,Yes,Winter,No
321956,2020-03-10 18:03:08,10,7739,4,612.28,0.1,Credit Card,Location C,19,Saturday,33,8,6757.7,516,20% Off,Yes,Spring,Yes


## Step 6 : Check Dataset Schema

In this step, I am checking the schema of each historical dataset.

All columns should be loaded as StringType because no transformation has been performed yet.

In [0]:
# ============================================================
# Step 6 : Check Dataset Schema
# ============================================================

# Display the schema of all historical datasets.

print("Customer Dataset Schema")
customer_df.printSchema()

print("\nProduct Dataset Schema")
product_df.printSchema()

print("\nSales Dataset Schema")
sales_df.printSchema()

Customer Dataset Schema
root
 |-- customer_id: string (nullable = true)
 |-- age: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- income_bracket: string (nullable = true)
 |-- loyalty_program: string (nullable = true)
 |-- membership_years: string (nullable = true)
 |-- churned: string (nullable = true)
 |-- marital_status: string (nullable = true)
 |-- number_of_children: string (nullable = true)
 |-- education_level: string (nullable = true)
 |-- occupation: string (nullable = true)
 |-- customer_zip_code: string (nullable = true)
 |-- customer_city: string (nullable = true)
 |-- customer_state: string (nullable = true)


Product Dataset Schema
root
 |-- product_id: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- product_brand: string (nullable = true)
 |-- product_category: string (nullable = true)
 |-- product_rating: string (nullable = true)
 |-- product_review_count: string (nullable = true)
 |-- product_stock: string (nullable = true

## Step 7 : Check Dataset Size

In this step, I am checking the number of records available in each historical dataset.

This helps verify that all datasets have been loaded completely before moving to the Landing layer.

In [0]:
# ============================================================
# Step 7 : Check Dataset Size
# ============================================================

# Count the number of records in each historical dataset.

print("Customer Records :", customer_df.count())
print("Product Records  :", product_df.count())
print("Sales Records    :", sales_df.count())

Customer Records : 1052
Product Records  : 1043
Sales Records    : 1002


## Step 8 : Check Missing Values

In this step, I am checking whether any columns contain missing (NULL) values.

This helps identify incomplete records before moving to the Landing Layer.

In [0]:
# ============================================================
# Step 8 : Check Missing Values
# ============================================================

from pyspark.sql.functions import col, when, count

# Check missing values in Customer Dataset

print("Customer Dataset")

display(
    customer_df.select([
        count(when(col(c).isNull(), c)).alias(c)
        for c in customer_df.columns
    ])
)

# Check missing values in Product Dataset

print("Product Dataset")

display(
    product_df.select([
        count(when(col(c).isNull(), c)).alias(c)
        for c in product_df.columns
    ])
)

# Check missing values in Sales Dataset

print("Sales Dataset")

display(
    sales_df.select([
        count(when(col(c).isNull(), c)).alias(c)
        for c in sales_df.columns
    ])
)

Customer Dataset


customer_id,age,gender,income_bracket,loyalty_program,membership_years,churned,marital_status,number_of_children,education_level,occupation,customer_zip_code,customer_city,customer_state
0,0,1,1,0,0,0,0,0,0,1,0,1,0


Product Dataset


product_id,product_name,product_brand,product_category,product_rating,product_review_count,product_stock,product_return_rate,product_size,product_weight,product_color,product_material,product_manufacture_date,product_expiry_date,product_shelf_life,unit_price
0,0,1,0,1,0,0,0,0,0,1,0,0,0,0,1


Sales Dataset


transaction_id,transaction_date,customer_id,product_id,quantity,unit_price,discount_applied,payment_method,store_location,transaction_hour,day_of_week,week_of_year,month_of_year,total_sales,promotion_id,promotion_type,holiday_season,season,weekend
0,26,0,0,0,25,27,107,91,33,63,35,18,28,0,129,154,95,191


## Step 9 : Check Duplicate Records

In this step, I am checking whether duplicate records are present in the historical datasets.

This helps me understand the quality of the raw data before moving to the Landing Layer.

No duplicate records will be removed in this notebook. Data cleaning and deduplication will be performed in the Silver Layer.

In [0]:
# ============================================================
# Step 9 : Check Duplicate Records
# ============================================================

# Calculate total records and unique records for each dataset.

customer_total = customer_df.count()
customer_unique = customer_df.dropDuplicates().count()

product_total = product_df.count()
product_unique = product_df.dropDuplicates().count()

sales_total = sales_df.count()
sales_unique = sales_df.dropDuplicates().count()


print("Customer Dataset")
print(f"Total Records     : {customer_total}")
print(f"Unique Records    : {customer_unique}")
print(f"Duplicate Records : {customer_total - customer_unique}")


print("\nProduct Dataset")
print(f"Total Records     : {product_total}")
print(f"Unique Records    : {product_unique}")
print(f"Duplicate Records : {product_total - product_unique}")


print("\nSales Dataset")
print(f"Total Records     : {sales_total}")
print(f"Unique Records    : {sales_unique}")
print(f"Duplicate Records : {sales_total - sales_unique}")

Customer Dataset
Total Records     : 1052
Unique Records    : 1051
Duplicate Records : 1

Product Dataset
Total Records     : 1043
Unique Records    : 1042
Duplicate Records : 1

Sales Dataset
Total Records     : 1002
Unique Records    : 1000
Duplicate Records : 2


## Step 10 : Display Dataset Summary

In this step, I am creating a summary of all historical datasets.

The summary includes:

- Number of rows
- Number of columns

This provides a quick overview of the datasets before moving to the next step.

In [0]:
# ============================================================
# Step 10 : Display Dataset Summary
# ============================================================

summary = [
    ("Customer", customer_df.count(), len(customer_df.columns)),
    ("Product", product_df.count(), len(product_df.columns)),
    ("Sales", sales_df.count(), len(sales_df.columns))
]

summary_df = spark.createDataFrame(
    summary,
    ["Dataset", "Rows", "Columns"]
)

display(summary_df)

Dataset,Rows,Columns
Customer,1052,14
Product,1043,16
Sales,1002,19


## Step 11 : Read Incremental Datasets

In this step, I am loading the incremental datasets into PySpark DataFrames.

At this stage, I am only reading the data.

No cleaning or transformation is performed in this step.

In [0]:
# ============================================================
# Step 11 : Read Incremental Datasets
# ============================================================

# Read Customer Incremental Dataset

customer_incremental_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", False)
    .csv(customer_incremental_path)
)

# Read Product Incremental Dataset

product_incremental_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", False)
    .csv(product_incremental_path)
)

# Read Sales Incremental Dataset

sales_incremental_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", False)
    .csv(sales_incremental_path)
)

print("Incremental datasets loaded successfully.")

Incremental datasets loaded successfully.


## Step 12 : Preview Incremental Datasets

In this step, I am displaying the incremental datasets to verify that they have been loaded successfully.

In [0]:
# ============================================================
# Step 12 : Preview Incremental Datasets
# ============================================================

# Display Customer Incremental Dataset

display(customer_incremental_df)

# Display Product Incremental Dataset

display(product_incremental_df)

# Display Sales Incremental Dataset

display(sales_incremental_df)

customer_id,age,gender,income_bracket,loyalty_program,membership_years,churned,marital_status,number_of_children,education_level,occupation,customer_zip_code,customer_city,customer_state,surrogate_key,version,effective_start_date,effective_end_date,is_current
1,56,Other,High,No,0,No,Divorced,3,Bachelor's,Self-Employed,37848,Old_City_1,Old_State_1,501,1,2020-01-01,2021-12-31,False
1,56,Other,High,No,0,No,Divorced,3,Bachelor's,Self-Employed,37848,New York,State NY,1,2,2022-01-01,null,True
2,69,Female,Medium,No,2,No,Married,2,PhD,Unemployed,44896,Old_City_2,Old_State_2,502,1,2020-01-01,2021-12-31,False
2,69,Female,Medium,No,2,No,Married,2,PhD,Unemployed,44896,Los Angeles,State CA,2,2,2022-01-01,null,True
3,46,Female,Low,No,5,No,Married,3,Bachelor's,Self-Employed,11816,City B,State X,3,1,2022-01-01,null,True
4,32,Female,Low,No,0,No,Divorced,2,Master's,Employed,78604,Old_City_3,Old_State_3,503,1,2020-01-01,2021-12-31,False
4,32,Female,Low,No,0,No,Divorced,2,Master's,Employed,78604,Chicago,State IL,4,2,2022-01-01,null,True
5,60,Female,Low,Yes,7,Yes,Divorced,2,Bachelor's,Employed,17760,City B,State Z,5,1,2022-01-01,null,True
6,25,Other,Medium,Yes,4,Yes,Divorced,0,Bachelor's,Unemployed,54549,City D,State Z,6,1,2022-01-01,null,True
7,78,Male,High,No,0,No,Single,2,Master's,Retired,76235,City D,State X,7,1,2022-01-01,null,True


product_id,product_name,product_brand,product_category,product_rating,product_review_count,product_stock,product_return_rate,product_size,product_weight,product_color,product_material,product_manufacture_date,product_expiry_date,product_shelf_life,unit_price,last_updated
1480,Product D,Brand Y,Electronics,2.8,560,98,0.4,Small,4.61,Red,Metal,2019-08-04 01:47:01,2022-05-28 14:54:02,250,54.69,2026-04-17
1597,Product C,Brand X,Groceries,4.7,413,80,0.3,Medium,0.84,Blue,Metal,2019-10-23 19:59:17,2022-12-19 08:04:41,180,817.76,2026-04-17
5142,Product B,Brand X,Toys,4.6,312,14,0.08,Medium,0.23,Green,Plastic,2018-05-12 08:00:29,2023-02-01 12:15:07,131,270.3,2026-04-17
8447,Product A,Brand Z,Toys,1.4,110,119,0.09,Large,4.37,Blue,Wood,2019-11-15 16:17:29,2023-02-05 11:46:57,16,602.62,2026-04-17
6025,Product C,Brand X,Clothing,3.8,172,25,0.39,Small,1.68,Red,Metal,2019-08-27 02:58:19,2023-10-05 08:13:07,57,785.29,2026-04-17
1883,Product A,Brand Y,Clothing,3.3,678,74,0.38,Large,0.26,Green,Glass,2019-02-09 22:23:36,2022-10-31 01:48:25,89,751.17,2026-04-17
7781,Product D,Brand Y,Toys,2.7,434,70,0.09,Medium,9.12,White,Plastic,2018-12-14 12:35:24,2023-11-04 17:03:25,316,374.08,2026-04-17
8642,Product B,Brand Y,Groceries,2.4,868,60,0.26,Medium,0.58,Blue,Wood,2018-01-06 22:13:14,2022-01-24 05:50:56,360,17.7,2026-04-17
7193,Product B,Brand Z,Groceries,4.4,392,45,0.28,Medium,6.52,Blue,Wood,2018-12-25 09:49:50,2022-10-04 03:00:26,77,458.85,2026-04-17
7739,Product A,Brand X,Clothing,3.6,356,61,0.03,Large,8.38,Red,Plastic,2018-02-27 19:00:28,2022-02-20 12:19:26,10,612.28,2026-04-17


transaction_id,transaction_date,customer_id,product_id,quantity,unit_price,discount_applied,payment_method,store_location,transaction_hour,day_of_week,week_of_year,month_of_year,total_sales,promotion_id,promotion_type,holiday_season,season,weekend
5002714,2023-02-04 19:23:44,334,258,8,987.33,0.35,Credit Card,null,19,Monday,5,2,null,100,20% Off,No,Fall,No
5256471,2023-02-15 02:02:05,73,5708,2,519.75,0.41,Cash,Location D,2,Monday,7,2,613.31,358,null,null,Summer,null
6989689,2023-09-05 07:34:41,195,8719,2,806.52,0.47,Mobile Payment,Location A,7,null,36,9,854.91,475,20% Off,null,null,Yes
3145707,2023-11-20 04:12:02,831,5830,7,193.05,0.35,Credit Card,null,4,Thursday,47,11,878.38,967,Flash Sale,Yes,null,null
7253487,2023-01-26 14:42:18,379,5489,8,null,0.38,Mobile Payment,Location D,14,null,4,1,2821.02,476,Buy One Get One Free,No,Spring,null
7583179,null,601,5080,3,239.94,0.48,null,Location C,4,Wednesday,17,8,374.31,160,null,Yes,null,null
9670485,2023-05-14 17:49:31,564,650,5,675.61,0.41,Debit Card,Location C,17,Saturday,null,5,1993.05,915,Buy One Get One Free,Yes,Winter,null
4558281,2023-01-09 00:58:09,440,1197,1,769.85,0.0,null,Location A,0,Wednesday,2,1,769.85,219,20% Off,Yes,Fall,null
9479252,2023-04-29 21:50:19,41,4911,8,113.05,0.4,null,null,21,Tuesday,17,4,542.64,980,null,Yes,Summer,No
9973800,2022-03-26 08:38:14,936,6833,2,434.36,0.27,Credit Card,Location A,8,Sunday,12,3,634.17,884,Flash Sale,No,Winter,Yes


## Step 13 : Check Incremental Dataset Size

In this step, I am checking the number of records available in each incremental dataset.

This confirms that all incremental datasets have been loaded successfully before moving to the audit validation.

In [0]:
# ============================================================
# Step 13 : Check Incremental Dataset Size
# ============================================================

# Count the number of records in each incremental dataset.

print("Customer Incremental Records :", customer_incremental_df.count())
print("Product Incremental Records  :", product_incremental_df.count())
print("Sales Incremental Records    :", sales_incremental_df.count())

Customer Incremental Records : 1053
Product Incremental Records  : 1041
Sales Incremental Records    : 1000


## Step 14 : Read Audit Datasets

In this step, I am loading the audit datasets into PySpark DataFrames.

These audit files contain the expected record count for each dataset. The audit information will be used later to compare the expected and actual record counts and validate the data before moving to the next layer.

In [0]:
# ============================================================
# Step 14 : Read Audit Datasets
# ============================================================

# Read Customer Audit Datasets

customer_historical_audit_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", False)
    .csv(customer_historical_audit_path)
)

customer_incremental_audit_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", False)
    .csv(customer_incremental_audit_path)
)

# Read Product Audit Datasets

product_historical_audit_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", False)
    .csv(product_historical_audit_path)
)

product_incremental_audit_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", False)
    .csv(product_incremental_audit_path)
)

# Read Sales Audit Datasets

sales_historical_audit_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", False)
    .csv(sales_historical_audit_path)
)

sales_incremental_audit_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", False)
    .csv(sales_incremental_audit_path)
)

print("Audit datasets loaded successfully.")

Audit datasets loaded successfully.


## Step 15 : Check Audit Dataset Size

In this step, I am checking the number of records available in each audit dataset.

This helps verify that all audit files have been loaded successfully before performing the audit validation.

In [0]:
# ============================================================
# Step 15 : Check Audit Dataset Size
# ============================================================

# Count the number of records in each audit dataset.

print("Customer Historical Audit   :", customer_historical_audit_df.count())
print("Customer Incremental Audit  :", customer_incremental_audit_df.count())

print("Product Historical Audit    :", product_historical_audit_df.count())
print("Product Incremental Audit   :", product_incremental_audit_df.count())

print("Sales Historical Audit      :", sales_historical_audit_df.count())
print("Sales Incremental Audit     :", sales_incremental_audit_df.count())

Customer Historical Audit   : 1
Customer Incremental Audit  : 1
Product Historical Audit    : 1
Product Incremental Audit   : 1
Sales Historical Audit      : 1
Sales Incremental Audit     : 1


## Step 16 : Compare Historical and Incremental Datasets

In this step, I am comparing the number of records in the historical and incremental datasets.

This comparison helps verify that both datasets have been loaded successfully and provides a quick overview before moving to the Landing Layer.

In [0]:
# ============================================================
# Step 16 : Compare Historical and Incremental Datasets
# ============================================================

comparison = [
    ("Customer", customer_df.count(), customer_incremental_df.count()),
    ("Product", product_df.count(), product_incremental_df.count()),
    ("Sales", sales_df.count(), sales_incremental_df.count())
]

comparison_df = spark.createDataFrame(
    comparison,
    ["Dataset", "Historical Records", "Incremental Records"]
)

display(comparison_df)

Dataset,Historical Records,Incremental Records
Customer,1052,1053
Product,1043,1041
Sales,1002,1000


## Step 17 : Create Landing Layer

In this step, I am creating the Landing Layer for this project.

The historical and incremental CSV datasets will be converted into Parquet format and stored in the Landing Layer.

Parquet is a columnar storage format that improves storage efficiency and provides faster read performance for downstream processing.

The Landing Layer will be used as the source for the Bronze Layer.

In [0]:
# ============================================================
# Step 17 : Convert Historical CSV Files to Parquet
# ============================================================

# Convert Customer Historical Dataset

customer_df.write.mode("overwrite").parquet(
    f"{LANDING_PATH}/historical/customer"
)

# Convert Product Historical Dataset

product_df.write.mode("overwrite").parquet(
    f"{LANDING_PATH}/historical/product"
)

# Convert Sales Historical Dataset

sales_df.write.mode("overwrite").parquet(
    f"{LANDING_PATH}/historical/sales"
)

print("Historical datasets converted to Parquet successfully.")

Historical datasets converted to Parquet successfully.


In [0]:
# ============================================================
# Step 18 : Convert Incremental CSV Files to Parquet
# ============================================================

# Convert Customer Incremental Dataset

customer_incremental_df.write.mode("overwrite").parquet(
    f"{LANDING_PATH}/incremental/customer"
)

# Convert Product Incremental Dataset

product_incremental_df.write.mode("overwrite").parquet(
    f"{LANDING_PATH}/incremental/product"
)

# Convert Sales Incremental Dataset

sales_incremental_df.write.mode("overwrite").parquet(
    f"{LANDING_PATH}/incremental/sales"
)

print("Incremental datasets converted to Parquet successfully.")

Incremental datasets converted to Parquet successfully.


## Step 19 : Verify Landing Layer

In this step, I am verifying that all Parquet files have been created successfully in the Landing Layer.

This ensures that the Landing Layer is ready for the Bronze Layer.

In [0]:
# ============================================================
# Step 19 : Verify Landing Layer
# ============================================================

print("Historical Landing Files")

display(dbutils.fs.ls(f"{LANDING_PATH}/historical"))

print("Incremental Landing Files")

display(dbutils.fs.ls(f"{LANDING_PATH}/incremental"))

Historical Landing Files


path,name,size,modificationTime
dbfs:/Volumes/workspace/default/apex_retail_volume/landing/historical/customer/,customer/,0,1786192126294
dbfs:/Volumes/workspace/default/apex_retail_volume/landing/historical/product/,product/,0,1786192126294
dbfs:/Volumes/workspace/default/apex_retail_volume/landing/historical/sales/,sales/,0,1786192126294


Incremental Landing Files


path,name,size,modificationTime
dbfs:/Volumes/workspace/default/apex_retail_volume/landing/incremental/customer/,customer/,0,1786192127147
dbfs:/Volumes/workspace/default/apex_retail_volume/landing/incremental/product/,product/,0,1786192127147
dbfs:/Volumes/workspace/default/apex_retail_volume/landing/incremental/sales/,sales/,0,1786192127147


## Step 20 : Landing Layer Summary

The Raw and Landing Layer has been completed successfully.

The following tasks were completed:

- Loaded historical datasets
- Loaded incremental datasets
- Loaded audit datasets
- Validated dataset information
- Converted CSV files into Parquet format
- Verified Landing Layer files

The Landing Layer is now ready for the Bronze Layer.

In [0]:
# ============================================================
# Step 20 : Convert Incremental CSV Files to Parquet
# ============================================================

# Save Customer Incremental Dataset

customer_incremental_df.write.mode("overwrite").parquet(
    f"{LANDING_PATH}/incremental/customer"
)

# Save Product Incremental Dataset

product_incremental_df.write.mode("overwrite").parquet(
    f"{LANDING_PATH}/incremental/product"
)

# Save Sales Incremental Dataset

sales_incremental_df.write.mode("overwrite").parquet(
    f"{LANDING_PATH}/incremental/sales"
)

print("Incremental datasets converted to Parquet successfully.")

Incremental datasets converted to Parquet successfully.


## Step 21 : Verify Parquet Files

In this step, I am checking whether all historical and incremental datasets have been successfully converted into Parquet format.

This helps verify that the Landing Layer has been created successfully and is ready for the Bronze Layer.

In [0]:
# ============================================================
# Step 21 : Verify Parquet Files
# ============================================================

print("Historical Parquet Files")

display(dbutils.fs.ls(f"{LANDING_PATH}/historical"))

print("Incremental Parquet Files")

display(dbutils.fs.ls(f"{LANDING_PATH}/incremental"))

Historical Parquet Files


path,name,size,modificationTime
dbfs:/Volumes/workspace/default/apex_retail_volume/landing/historical/customer/,customer/,0,1786192246438
dbfs:/Volumes/workspace/default/apex_retail_volume/landing/historical/product/,product/,0,1786192246438
dbfs:/Volumes/workspace/default/apex_retail_volume/landing/historical/sales/,sales/,0,1786192246438


Incremental Parquet Files


path,name,size,modificationTime
dbfs:/Volumes/workspace/default/apex_retail_volume/landing/incremental/customer/,customer/,0,1786192247177
dbfs:/Volumes/workspace/default/apex_retail_volume/landing/incremental/product/,product/,0,1786192247177
dbfs:/Volumes/workspace/default/apex_retail_volume/landing/incremental/sales/,sales/,0,1786192247177


## Step 22 : Validate Landing Data Using Audit Files

In this step, I am validating the Landing Layer using the audit files.

Each audit file contains the expected number of records for a dataset. I will compare the expected row count with the actual row count from the Landing Layer.

If the expected and actual row counts match, the validation status will be marked as **PASS**. Otherwise, it will be marked as **FAIL**.

This validation confirms that all datasets have been loaded correctly before proceeding to the Bronze Layer.

In [0]:
# ============================================================
# Step 22 : Read Expected Row Counts from Audit Files
# ============================================================

customer_hist_expected = int(customer_historical_audit_df.first()["row_count"])
customer_inc_expected = int(customer_incremental_audit_df.first()["row_count"])

product_hist_expected = int(product_historical_audit_df.first()["row_count"])
product_inc_expected = int(product_incremental_audit_df.first()["row_count"])

sales_hist_expected = int(sales_historical_audit_df.first()["row_count"])
sales_inc_expected = int(sales_incremental_audit_df.first()["row_count"])

print("Expected row counts loaded successfully.")

Expected row counts loaded successfully.


## Step 23 : Compare Expected and Actual Row Counts

In this step, I am comparing the expected row counts from the audit files with the actual row counts from the historical and incremental datasets.

If the expected and actual row counts are the same, the validation status will be marked as **PASS**. Otherwise, it will be marked as **FAIL**.

This validation ensures that all datasets have been loaded correctly into the Landing Layer before moving to the Bronze Layer.

In [0]:
# ============================================================
# Step 23 : Compare Expected and Actual Row Counts
# ============================================================

audit_report = [

    (
        "Customer Historical",
        customer_hist_expected,
        customer_df.count(),
        "PASS" if customer_hist_expected == customer_df.count() else "FAIL"
    ),

    (
        "Customer Incremental",
        customer_inc_expected,
        customer_incremental_df.count(),
        "PASS" if customer_inc_expected == customer_incremental_df.count() else "FAIL"
    ),

    (
        "Product Historical",
        product_hist_expected,
        product_df.count(),
        "PASS" if product_hist_expected == product_df.count() else "FAIL"
    ),

    (
        "Product Incremental",
        product_inc_expected,
        product_incremental_df.count(),
        "PASS" if product_inc_expected == product_incremental_df.count() else "FAIL"
    ),

    (
        "Sales Historical",
        sales_hist_expected,
        sales_df.count(),
        "PASS" if sales_hist_expected == sales_df.count() else "FAIL"
    ),

    (
        "Sales Incremental",
        sales_inc_expected,
        sales_incremental_df.count(),
        "PASS" if sales_inc_expected == sales_incremental_df.count() else "FAIL"
    )

]

audit_report_df = spark.createDataFrame(
    audit_report,
    ["Dataset", "Expected Rows", "Actual Rows", "Validation Status"]
)

display(audit_report_df)

Dataset,Expected Rows,Actual Rows,Validation Status
Customer Historical,1052,1052,PASS
Customer Incremental,1053,1053,PASS
Product Historical,1043,1043,PASS
Product Incremental,1041,1041,PASS
Sales Historical,1002,1002,PASS
Sales Incremental,1000,1000,PASS


## Step 24 : Landing Layer Summary

The Raw and Landing Layer has been completed successfully.

The following tasks were completed in this notebook:

- Verified the available dataset files.
- Loaded historical datasets.
- Loaded incremental datasets.
- Loaded audit datasets.
- Checked the dataset schema.
- Verified record counts.
- Checked missing values.
- Checked duplicate records.
- Compared historical and incremental datasets.
- Converted CSV files into Parquet format.
- Validated the Landing Layer using audit files.
- Generated a PASS / FAIL validation report.

The Landing Layer is now ready for the Bronze Layer, where the Parquet files will be converted into Delta Lake tables with metadata for further processing.

In [0]:
# ============================================================
# Step 24 : Landing Layer Summary
# ============================================================

print("Raw and Landing Layer completed successfully.")
print("Landing Layer is ready for the Bronze Layer.")

Raw and Landing Layer completed successfully.
Landing Layer is ready for the Bronze Layer.


## Notebook Summary

✔ Historical Data Loaded

✔ Incremental Data Loaded

✔ Audit Validation Completed

✔ Parquet Files Created

✔ Ready for Bronze Layer